[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helvecioneto/datascience/blob/main/praticas/pratica03-1-dbscan.ipynb)

# Aula Prática 03 · Exemplo 1 de 3: Clusterização com DBSCAN

**Disciplina:** Introdução à Ciência de Dados (NSA A110) · **Professor:** Helvecio Bezerra Leal Neto · UFOPA
**Data:** 29/10/2026, 19h às 22h · **Modalidade:** prática em equipes (2 a 3 pessoas)

Hoje são **três notebooks curtos**, um para cada técnica: **1.** clusterização com DBSCAN · **2.** árvore de decisão · **3.** redes neurais. Este é o **1**.

### Como usar este notebook
Leia cada bloco de texto **antes** de rodar a célula de código logo abaixo dele. Os ícones dizem o que fazer:

| Ícone | O que você faz |
|---|---|
| ▶️ **Rode** | A célula está pronta. Execute com `Shift + Enter` e leia a saída. |
| 🧭 **Complete** | Troque cada `___` pelo que o texto pede e rode. Se travar, abra o **💡 Ver resposta** logo abaixo, mas tente antes. |
| ✍️ **Responda** | Escreva a resposta da equipe na célula de texto indicada (dois cliques para editar). |

Rode as células **na ordem**, de cima para baixo. Se algo der errado: *Ambiente de execução → Reiniciar e executar tudo*.

### Objetivos
Ao final, a equipe será capaz de:
- **Explicar** com as próprias palavras como o DBSCAN forma grupos: vizinhança (`eps`), mínimo de vizinhos (`min_samples`) e ruído.
- **Comparar** DBSCAN e k-means em dados com formatos curvos.
- **Escolher** o `eps` olhando o resultado e **usar** o DBSCAN num mapa real para encontrar cidades.

### O que vocês já sabem e o que é novo

Na Aula 07 vimos a **clusterização**: juntar em grupos os dados **parecidos**, **sem rótulo** (aprendizado não supervisionado). A técnica foi o **k-means**, em que **nós** escolhemos o número de grupos (o `K`).

Hoje conhecemos outra técnica: o **DBSCAN**, que agrupa por **densidade**. Densidade é **quantos pontos há num pedaço do espaço**: uma rodinha cheia de gente é um lugar de alta densidade. A ideia é a de uma **festa**:
- quem está numa **rodinha** de conversa, cercado de gente, forma um **grupo**;
- quem está **sozinho num canto** não pertence a grupo nenhum: é **ruído**.

O DBSCAN só precisa de duas regras, que **nós** escolhemos:

| Parâmetro | Pergunta que ele responde | Na festa |
|---|---|---|
| `eps` | Até que distância um ponto é **vizinho** do outro? | o alcance da voz |
| `min_samples` | Quantos vizinhos (contando o próprio ponto) formam uma **rodinha**? | o tamanho mínimo da rodinha |

Com isso, cada ponto vira um de três tipos:
- **central**: tem pelo menos `min_samples` vizinhos. É o coração de um grupo;
- **borda**: tem poucos vizinhos, mas está **dentro do círculo** (de raio `eps`) de um ponto central. Entra no grupo dele;
- **ruído**: não é central nem está perto de um central. Recebe o rótulo **`-1`**.

Repare na grande diferença: **ninguém diz ao DBSCAN quantos grupos existem**. Ele descobre.

## Etapa 0 · Preparar o ambiente
▶️ **Rode** as duas células. A primeira importa as bibliotecas; a segunda cria duas funções que **desenham** os pontos (não precisam entender o código delas por dentro).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons, fetch_california_housing
from sklearn.metrics import pairwise_distances

SEMENTE = 42   # mesma semente = mesmos resultados para todas as equipes
print("Tudo pronto!")

In [ ]:
def mostrar_grupos(pontos, rotulos, titulo="", ax=None, tamanho=40):
    # pinta cada grupo de uma cor (27 cores, as 10 primeiras bem diferentes); o ruído (-1) aparece como um x preto
    ax = ax or plt.gca()
    ruido = rotulos == -1
    cores = list(plt.cm.tab10.colors) + list(plt.cm.Dark2.colors) + list(plt.cm.Set1.colors)
    ax.scatter(pontos[~ruido, 0], pontos[~ruido, 1], c=[cores[r % len(cores)] for r in rotulos[~ruido]], s=tamanho)
    ax.scatter(pontos[ruido, 0], pontos[ruido, 1], c="black", marker="x", s=tamanho, label="ruído (-1)")
    if ruido.any():
        ax.legend(loc="best")
    ax.set_title(titulo)


def mostrar_vizinhanca(pontos, eps):
    # desenha um círculo de raio eps em volta de cada ponto, com o número do ponto
    fig, ax = plt.subplots(figsize=(6, 6))
    for x, y in pontos:
        ax.add_patch(plt.Circle((x, y), eps, color="tab:blue", alpha=0.08))
    ax.scatter(pontos[:, 0], pontos[:, 1], color="tab:blue")
    for i, (x, y) in enumerate(pontos):
        ax.annotate(str(i), (x, y), xytext=(5, 5), textcoords="offset points", fontsize=12)
    ax.set_aspect("equal")
    ax.set_title(f"Círculo de vizinhança de cada ponto (eps = {eps})")
    plt.show()

print("Funções de desenho prontas!")

## Etapa 1 · A ideia com 13 pontos
Vamos começar com poucos pontos, para conseguir **conferir tudo com os olhos**. Cada ponto tem duas coordenadas, `x` e `y`.

Vamos usar `eps = 1.5` e `min_samples = 3`. O desenho mostra, em volta de cada ponto, um **círculo de raio 1.5**: tudo o que cai dentro do círculo é **vizinho** daquele ponto.

▶️ **Rode.**

In [ ]:
pontos = np.array([
    [1.0, 1.0], [1.5, 1.8], [2.0, 1.2], [1.2, 2.3], [2.3, 2.1],   # pontos 0 a 4
    [3.3, 2.6],                                                  # ponto 5
    [6.0, 5.0], [6.8, 5.4], [6.3, 6.1], [7.1, 6.3], [5.8, 5.9],   # pontos 6 a 10
    [3.8, 8.0], [9.0, 1.5],                                      # pontos 11 e 12
])
print("Temos", len(pontos), "pontos")

mostrar_vizinhanca(pontos, eps=1.5)

### Contando os vizinhos
O computador mede a **distância** de cada ponto a todos os outros e conta quantos estão a no máximo `eps` (o próprio ponto conta). Quem tem **pelo menos 3** é **central**. Na tabela, a linha `0` é o ponto 0, a linha `1` é o ponto 1, e assim por diante.

▶️ **Rode.**

In [ ]:
EPS = 1.5
MIN_SAMPLES = 3

distancias = pairwise_distances(pontos)        # tabela 13 × 13: distância de cada ponto a cada outro
vizinhos = (distancias <= EPS).sum(axis=1)     # quantos cabem no círculo de cada ponto

pd.DataFrame({"quem está no círculo": [list(np.where(linha <= EPS)[0]) for linha in distancias],
              "vizinhos (contando ele mesmo)": vizinhos,
              "é central? (pelo menos 3)": vizinhos >= MIN_SAMPLES})

✍️ **Responda** olhando a tabela e o desenho:
1. Quais pontos são **centrais**?
2. O ponto **5** não é central. Ele está dentro do círculo de algum ponto central? Então ele é **borda** ou **ruído**?
3. E os pontos **11** e **12**: são o quê?

**✍️ Respostas da equipe**

1.
2.
3.

### Agora o DBSCAN faz isso sozinho
No scikit-learn, o DBSCAN se usa em dois passos: **criar** o modelo com os parâmetros e chamar `fit_predict`, que **encontra os grupos e devolve o grupo de cada ponto**.

🧭 **Complete:** na linha que conta os pontos de ruído, coloquem o **número que o DBSCAN usa para marcar ruído** (está na introdução do notebook).

In [ ]:
modelo = DBSCAN(eps=1.5, min_samples=3)
rotulos = modelo.fit_predict(pontos)

print("Grupo de cada ponto:", rotulos)
print("Quantos viraram ruído:", (rotulos == ___).sum())   # compara cada rótulo com o número do ruído e conta os True
mostrar_grupos(pontos, rotulos, "DBSCAN com eps = 1.5 e min_samples = 3")
plt.show()

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
print("Quantos viraram ruído:", (rotulos == -1).sum())   # -1 = ruído
```
</details>

Como ler a saída: a lista mostra o grupo de cada ponto, na ordem (ponto 0, ponto 1, …). Os grupos são numerados a partir de `0`, e **`-1` quer dizer ruído**.

✍️ **Responda:**
1. Quantos grupos o DBSCAN encontrou? O resultado bate com o que vocês responderam acima?
2. Em algum momento nós dissemos ao DBSCAN **quantos grupos** queríamos? Comparem com o k-means da Aula 07.

**✍️ Respostas da equipe**

1.
2.

### E se mudarmos o `eps`?
▶️ **Rode** para ver o mesmo DBSCAN com três alcances diferentes: pequeno (0.5), o nosso (1.5) e grande (5).

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, eps in zip(eixos, [0.5, 1.5, 5]):
    rot = DBSCAN(eps=eps, min_samples=3).fit_predict(pontos)
    n_grupos = len(set(rot) - {-1})    # quantos rótulos diferentes existem, sem contar o -1 (ruído)
    mostrar_grupos(pontos, rot, f"eps = {eps}: {n_grupos} grupo(s), {(rot == -1).sum()} ruído(s)", ax=ax)
plt.show()

✍️ **Responda:** o que acontece quando o `eps` é **pequeno demais** (0.5)? E quando é **grande demais** (5)? Expliquem pensando no tamanho dos círculos.

**✍️ Resposta da equipe:**


## Etapa 2 · Um formato que o k-means não pega
O k-means agrupa os pontos em volta de um **centro**, como se os grupos fossem **bolas**. E quando os grupos têm outra forma?

▶️ **Rode** para criar os dados: duas **meias-luas** encaixadas (300 pontos) e mais **20 pontos soltos**, espalhados ao acaso.

In [ ]:
luas, _ = make_moons(n_samples=300, noise=0.06, random_state=SEMENTE)                      # duas meias-luas
soltos = np.random.RandomState(SEMENTE).uniform([-1.5, -1], [2.5, 1.5], size=(20, 2))  # 20 pontos ao acaso
X = np.vstack([luas, soltos])                                                           # junta tudo

print("Total de pontos:", len(X))
plt.figure(figsize=(7, 4.5))
plt.scatter(X[:, 0], X[:, 1], s=15, color="gray")
plt.title("Os dados: quantos grupos vocês enxergam?")
plt.show()

▶️ **Rode** o **k-means** com `K = 2`, como na Aula 07.

In [ ]:
rot_kmeans = KMeans(n_clusters=2, random_state=SEMENTE, n_init=10).fit_predict(X)

plt.figure(figsize=(7, 4.5))
mostrar_grupos(X, rot_kmeans, "k-means com K = 2")
plt.show()

Agora o **DBSCAN**, com `eps = 0.2` e `min_samples = 5`. (Por que 0.2? Vamos descobrir na Etapa 3.)

No desenho, os **20 pontos soltos** ganham um **círculo vermelho** em volta, para vocês saberem onde eles caíram.

🧭 **Complete:** passe para o `fit_predict` a variável que guarda **todos os pontos** desta etapa.

In [ ]:
rot_dbscan = DBSCAN(eps=0.2, min_samples=5).fit_predict(___)

print("Grupos encontrados:", len(set(rot_dbscan) - {-1}))
print("Pontos marcados como ruído:", (rot_dbscan == -1).sum())
print("Dos 20 pontos soltos, viraram ruído:", (rot_dbscan[300:] == -1).sum())   # [300:] = do ponto 300 até o fim: os 20 soltos

plt.figure(figsize=(7, 4.5))
mostrar_grupos(X, rot_dbscan, "DBSCAN com eps = 0.2 e min_samples = 5")
plt.scatter(soltos[:, 0], soltos[:, 1], s=160, facecolors="none", edgecolors="red", label="os 20 pontos soltos")
plt.legend()
plt.show()

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
rot_dbscan = DBSCAN(eps=0.2, min_samples=5).fit_predict(X)
```
</details>

✍️ **Responda:**
1. Qual dos dois métodos separou as duas meias-luas corretamente?
2. Por que o k-means errou? *(Dica: lembrem que ele procura grupos em volta de um centro.)*
3. Dos 20 pontos soltos (círculos vermelhos), quantos o DBSCAN marcou como ruído? Olhem os círculos vermelhos **sem** o x preto: onde eles caíram? Por que não viraram ruído?

**✍️ Respostas da equipe**

1.
2.
3.

## Etapa 3 · Como escolher o `eps`?
Não existe um `eps` certo para todos os dados: ele depende de **quão longe os pontos costumam estar uns dos outros** (nas luas, décimos; no mapa da Etapa 4, graus de latitude). O jeito mais simples é **testar alguns valores e olhar o desenho**.

▶️ **Rode** para ver as meias-luas com quatro valores de `eps` (sempre com `min_samples = 5`). Quando há muitos grupos, algumas cores ficam parecidas: **confiem no número do título**.

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(13, 8))
for ax, eps in zip(eixos.ravel(), [0.05, 0.1, 0.2, 0.5]):
    rot = DBSCAN(eps=eps, min_samples=5).fit_predict(X)
    n_grupos = len(set(rot) - {-1})
    mostrar_grupos(X, rot, f"eps = {eps}: {n_grupos} grupo(s), {(rot == -1).sum()} ruído(s)", ax=ax, tamanho=15)
plt.tight_layout()
plt.show()

✍️ **Responda:** qual `eps` deu o melhor resultado? Como vocês sabem, olhando o desenho?

**✍️ Resposta da equipe:**


## Etapa 4 · Uso real: onde ficam as cidades da Califórnia?
Agora um caso de verdade. A base **California Housing** vem do **censo dos EUA de 1990** e já está embutida no scikit-learn. Ela divide o estado da Califórnia em **20 640 quarteirões** (cada um com algumas centenas ou milhares de moradores), e de cada quarteirão sabemos a **latitude** e a **longitude**.

**A pergunta:** sem dizer o nome de nenhuma cidade, o DBSCAN consegue encontrar as **grandes regiões urbanas**? A ideia é simples: onde há **muitos quarteirões juntos**, há **muita gente**, ou seja, uma cidade.

▶️ **Rode** para baixar os dados (leva alguns segundos na primeira vez) e desenhar o mapa. Se aparecer um erro de download, chamem o professor.

In [ ]:
casas = fetch_california_housing(as_frame=True).frame   # base do censo, embutida no scikit-learn
mapa = casas[["Longitude", "Latitude"]].to_numpy()      # x = longitude, y = latitude

print("Quarteirões:", len(mapa))
plt.figure(figsize=(6, 7))
plt.scatter(mapa[:, 0], mapa[:, 1], s=1, color="gray")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.title("Cada ponto é um quarteirão da Califórnia")
plt.show()

Repararam que o mapa já tem o **formato do estado**? E que há manchas bem mais escuras, onde os pontos se amontoam?

Os parâmetros, traduzidos para o mapa:
- `eps = 0.2` grau de latitude/longitude, que dá cerca de **20 km**;
- `min_samples = 300` quarteirões.

Ou seja: *"um lugar só é cidade grande se, num raio de uns 20 km, houver pelo menos 300 quarteirões"*.

🧭 **Complete** com o `min_samples` e rode. Pode levar alguns segundos.

In [ ]:
modelo_mapa = DBSCAN(eps=0.2, min_samples=___)
rot_mapa = modelo_mapa.fit_predict(mapa)

print("Regiões encontradas:", len(set(rot_mapa) - {-1}))
print(f"Quarteirões fora de qualquer região (ruído): {(rot_mapa == -1).mean():.0%}")

plt.figure(figsize=(6, 7))
mostrar_grupos(mapa, rot_mapa, "Regiões densas encontradas pelo DBSCAN", tamanho=2)
for g in set(rot_mapa) - {-1}:                       # escreve o número do grupo no centro de cada região
    lon, lat = mapa[rot_mapa == g].mean(axis=0)
    plt.annotate(str(g), (lon, lat), fontsize=14, fontweight="bold")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.show()

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
modelo_mapa = DBSCAN(eps=0.2, min_samples=300)
```
</details>

▶️ **Rode** para ver o **tamanho** e o **centro** (latitude e longitude médias) de cada região, da maior para a menor.

In [ ]:
regioes = pd.DataFrame({"longitude": mapa[:, 0], "latitude": mapa[:, 1], "grupo": rot_mapa})
regioes = regioes[regioes["grupo"] != -1]     # deixa o ruído de fora

resumo = regioes.groupby("grupo").agg(quarteiroes=("grupo", "size"),
                                      latitude_media=("latitude", "mean"),
                                      longitude_media=("longitude", "mean"))
resumo.sort_values("quarteiroes", ascending=False).round(2)

Para dar nome às regiões, comparem o centro de cada uma com as coordenadas de algumas cidades da Califórnia:

| Cidade | Latitude | Longitude |
|---|---|---|
| Los Angeles | 34.05 | -118.24 |
| San Francisco (Baía de São Francisco) | 37.77 | -122.42 |
| San Diego | 32.72 | -117.16 |
| Sacramento (a capital) | 38.58 | -121.49 |
| Fresno | 36.74 | -119.79 |
| Stockton | 37.96 | -121.29 |

**Dica:** para cada região, procurem a cidade com latitude **e** longitude a menos de 0.5 do centro. A longitude é negativa só porque fica a oeste: comparem -118 com -118. O número do grupo também está escrito no mapa.

✍️ **Responda:**
1. Deem o **nome de uma cidade** a cada região da tabela, da maior para a menor.
2. O que devem ser os quarteirões marcados como **ruído** no mapa?
3. Por que seria difícil fazer esta análise com o **k-means**? *(Dica: vocês saberiam o `K` antes? E o que o k-means faria com um quarteirão no meio do deserto?)*

**✍️ Respostas da equipe**

1.
2.
3.

> **k-means × DBSCAN, lado a lado**
>
> | | k-means | DBSCAN |
> |---|---|---|
> | Número de grupos | **nós** escolhemos (`K`) | **ele descobre** |
> | Formato dos grupos | bolas, em volta de um centro | **qualquer formato** |
> | Pontos isolados | sempre entram em algum grupo | viram **ruído** (`-1`) |
> | O que ajustamos | `K` | `eps` e `min_samples` |
>
> Onde o DBSCAN é usado: agrupar pontos de **GPS** e de **mapas** (focos de queimada, ocorrências policiais, paradas de ônibus), achar **anomalias** (o ruído: uma compra estranha no cartão, um sensor com defeito) e separar objetos em **imagens**.

## 🏁 Desafio extra (para quem terminar antes)
Copiem a célula do DBSCAN do mapa para uma célula nova e testem `min_samples = 100` e depois `min_samples = 600` (mantendo `eps = 0.2`). Quantas regiões aparecem em cada caso? O ruído aumenta ou diminui? Por quê?

**✍️ Resposta da equipe:**


## Encerramento
**O que fizemos:**
1. Vimos que o DBSCAN agrupa por **densidade**: pontos **centrais**, de **borda** e **ruído**.
2. Ajustamos `eps` (alcance) e `min_samples` (tamanho mínimo da rodinha) e vimos o efeito de cada um.
3. Comparamos com o k-means: o DBSCAN **não precisa do `K`**, aceita **qualquer formato** e separa o **ruído**.
4. Encontramos as grandes cidades da Califórnia **sem dar o nome de nenhuma**.

➡️ **Próximo:** abram o notebook **2 · Árvore de decisão**.